資料來源:
Invent連結： 
kaggle.com/bhanupratapbiswas/inventory-analysis-case-study
PwC 是 PricewaterhouseCoopers 的縮寫，全球四大會計師事務所（Big Four）之一 。
數據集的業務邏輯設計符合真實審計和財務分析標準，欄位設計反映的是真實企業的採購流程，


所有 27 個 Result CSV 已全部讀取完畢。現在對資料做完整分析解讀。

***

## 📊 探索分析完整解讀（模組 1–7）

***

### 模組 1：資料集規模概覽

| 表 | 行數 | 說明 |
|---|---|---|
| `stg.sales` | **1,048,575** | 銷售明細（注意：正好是 Excel 行數上限，資料可能被截斷⚠️） |
| `stg.purchases` | **2,372,474** | 採購明細 |
| `stg.beg_inv` | **206,529** | 期初庫存 |
| `stg.end_inv` | **224,489** | 期末庫存 |
| `stg.purchase_prices` | **12,261** | 產品主檔 |
| `stg.invoice_purchases` | **5,543** | 發票彙總 |

**🚨 關鍵發現 1 — Sales 資料截斷問題**
`stg.sales` 有 **1,048,575 行 = 2^20 - 1**，這正是 Excel `.xlsx` 的最大行數上限。原始 Kaggle 資料集的 Sales 表極可能遠超此數，**原始 CSV 在 Excel 中開啟後另存時被截斷**。Sales 日期範圍也只有 **2016-01-01 至 2016-02-29**（僅 60 天），而採購覆蓋全年，印證截斷。**ETL 必須從 Kaggle 原始來源重新下載完整 CSV。**

***

### 模組 2：NULL 掃描 — 全部乾淨 ✅

| 表 | 總行數 | NULL 欄位數 | 結論 |
|---|---|---|---|
| `stg.sales` | 1,048,575 | **0** | 完全乾淨 |
| `stg.purchases` | 2,372,474 | **0** | 完全乾淨 |
| `stg.beg_inv` | 206,529 | **0** | 完全乾淨 |
| `stg.end_inv` | 224,489 | **0** | 完全乾淨 |
| `stg.purchase_prices` | 12,261 | **0** | 完全乾淨 |

所有表**無 NULL 值**，資料匯入完整。ETL 清洗重點可轉移至邏輯錯誤和業務規則驗證。

***

### 模組 3：維度分佈

| 維度 | 數量 | 含義 |
|---|---|---|
| 門市數 | **79** 間（67 城市） | 多城市、部分城市有多間門市 |
| 品牌數 | **12,261** | SKU 非常多樣化 |
| 商品描述 | **11,115**（57 種 Size） | 品牌數 > 描述數，少數品牌有多個 Size |
| Classification | Class 2: **70.9%**、Class 1: **29.1%** | 酒類行業 Class 2 = 葡萄酒/啤酒，Class 1 = 烈酒 |
| 供應商（採購/發票） | **126** 家 | 銷售側只有 **116** 家，10 家供應商只進貨不售 |

***

### 模組 4：Join 連接性驗證

| Join 關係 | 結果 | 解讀 |
|---|---|---|
| BegInv ↔ EndInv | 84.72% 匹配，31,553 僅期初，49,513 僅期末 | **正常**：年內有新品引入（49K）及商品退場（31K） |
| Sales ↔ BegInv | **98.31%** 匹配（2,872 條不匹配） | 極佳，少量銷售品項在期初不存在（可能期中新增）|
| Purchases ↔ PurchasePrices | **100%** 匹配 ✅ | 採購品牌均有主檔定價，COGS 計算完全可靠 |
| Sales ↔ PurchasePrices | **100%** 匹配 ✅ | 所有銷售品牌均有採購成本，KPI 計算零缺口 |

**🟢 Join 完整性極佳**，可直接進行 Star Schema 建模。

***

### 模組 5：異常值偵測

| 檢查項目 | 結果 | 處理建議 |
|---|---|---|
| 負數庫存 | **0** 筆 ✅ | 無需處理 |
| 零庫存（期初） | **6,044** 筆 | 正常（季節性/未開賣商品） |
| 零庫存（期末） | **7,230** 筆 | 可能缺貨，進 Stockout 計算 |
| 銷售金額範圍 | $0.49 ~ $13,279.97，中位數 $17.99 | 正常，無負值 |
| 採購單價核對 | **0** 筆不一致 ✅ | PurchasePrice × Qty = Dollars 完全吻合 |
| 日期邏輯錯誤 | **recv_before_po: 0, pay_before_inv: 0** ✅ | 無邏輯錯誤 |
| **neg_zero_dollars** | **153 筆** ⚠️ | 採購金額為 0，需調查（免費樣品？退貨補正？）|
| 重複主鍵 | **0** 筆 ✅ | 所有表主鍵唯一 |

**🚨 關鍵發現 2 — 153 筆 dollars = 0 的採購記錄**
須在 ODS 清洗層標記或排除，避免污染 COGS 計算。

***

### 模組 6：KPI 預先可行性試算

**Lead Time 分佈（採購前置時間）：**

| 指標 | 天數 |
|---|---|
| 最短 | 3 天 |
| P25 | 6 天 |
| **中位數（P50）** | **8 天** |
| 平均 | 7.62 天 |
| P75 | 9 天 |
| P95 | 11 天 |
| 最長 | 14 天 |
| 標準差 | 2.21 天 |

**非常穩定的供應鏈**（標準差僅 2.21 天），Reorder Point 計算可用此數據直接代入。

**毛利結構（各門市）：**
所有門市毛利率集中在 **30–35%**（最高 34.78%，最低 30.42%），非常穩定一致，說明定價機制統一。

***

### 模組 7：月度趨勢

**🚨 關鍵發現 3 — Sales 資料確認截斷**：

| 月份 | 銷售量 | 採購量 | 狀態 |
|---|---|---|---|
| 2016-01 | 2,194,959 | 2,232,313 | 有銷售數據 |
| 2016-02 | 256,210 | 2,182,428 | 有銷售數據（但量差異極大）|
| 2016-03 至 12 | **0** | 正常（240萬–330萬/月） | **銷售資料完全缺失** |

- **期末有 28,114 件潛在呆滯庫存**（期初有、期末有存量、但全年無銷售記錄）
- **31,553 件期初存在但期末消失**（已售罄或下架）
- **49,513 件期末新增品項**（年中引進新商品）

***

## 🎯 下一步決策

根據以上分析，我已掌握足夠資訊。請確認以下兩件事再進入 ETL：

**問題 1：Sales 截斷問題**
`stg.sales` 只有 Jan–Feb 2016 資料（1,048,575 行 = Excel 上限）。
- **選項 A**：你有完整版 Sales CSV（全年 12 個月）？ → 重新匯入再進行 ETL
- **選項 B**：先用現有 Jan–Feb 資料建立完整 ETL pipeline，之後換完整資料即可

**問題 2：153 筆 dollars=0 採購記錄**
- **選項 A**：直接排除（標記 `is_valid = false`）
- **選項 B**：保留並在 KPI 計算時用 `WHERE dollars > 0` 過濾

確認後我立刻輸出：
1. **完整 ELT SQL**（Staging → ODS 清洗 → DWH Star Schema）
2. **5 個 KPI View**（Inventory Turnover、DSI、Stockout Rate、Overstock%、Reorder Point）